# Travelling Salesman Problem (20 cities)

Formulation:

  State  : current city (one-hot) + visited mask (binary vector) → 40-dim input

  Action : next unvisited city (masked softmax)

  Reward : negative Euclidean distance of the full tour (given at end of episode)
  
  Episode ends when all cities have been visited and agent returns to start.

In [5]:
import numpy as np
import matplotlib.pyplot as plt

In [6]:
def softmax(x):
    # Converts logits into probabilities
    e = np.exp(x - np.max(x))
    return e / e.sum()

def relu(x):
    # Used in hidden layer
    return np.maximum(0, x)

def tour_length(cities, tour):
    # Total Euclidean tour length including return to start
    
    # Initialize the total distance
    dist = 0.0
    # Initialize number of cities in tour
    n = len(tour)

    # Iterate over all cities
    for i in range(n):
        # a gets the current city and b gets the next city
        a, b = cities[tour[i]], cities[tour[(i + 1) % n]]

        # Compute the Euclidean distance using L2 Norm
        dist += np.linalg.norm(a - b)
    return dist

# Policy Network 
We make a 2-layer Multi-Layer Perceptron since we cannot use any deep-learning library

In [7]:
class PolicyNetwork:
    # Input  : 40-dim (current city one-hot 20 + visited mask 20)
    # Hidden : 64 units, ReLU
    # Output : 20 logits → masked softmax → action probabilities
    
    def __init__(self, n_cities=20, hidden=64, lr=1e-3):
        # Stores values

        self.n = n_cities
        self.lr = lr

        # Input Dimension = 40
        in_dim = 2 * n_cities

    
        # Xavier initialization
        # Weight shape: (40 × 64)
        self.W1 = np.random.randn(in_dim, hidden) * np.sqrt(2.0 / in_dim)
        # Hidden Biases
        self.b1 = np.zeros(hidden)
        
        # Weight shape: (64 × 20)
        self.W2 = np.random.randn(hidden, n_cities) * np.sqrt(2.0 / hidden)
        self.b2 = np.zeros(n_cities)

    def forward(self, city_idx, visited_mask):
        # Returns (probabilities, cache

        # Build Input Vector
        x = np.zeros(2 * self.n)

        # One-hot encode current city
        x[city_idx] = 1.0

        # Append visited mask
        x[self.n:] = visited_mask

        # Hidden Layer, Apply ReLU
        h = relu(x @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2

        # Mask visited cities to force probability -> 0 for visited cities
        logits[visited_mask.astype(bool)] = -1e9
        probs = softmax(logits)
        return probs, (x, h, logits)


    # Action Selection
    def select_action(self, city_idx, visited_mask):

        # Forward Pass
        probs, cache = self.forward(city_idx, visited_mask)

        # Sample action stochastically for Policy Gradient
        action = np.random.choice(self.n, p=probs)

        # Compute Log Probability
        log_prob = np.log(probs[action] + 1e-9)
        return action, log_prob, cache

    # Policy Update
    def update(self, episode_log_probs, G):
        # G : scalar return (same for all steps since reward is given at end).
        
        # Accumulate gradients over all steps
        dW1 = np.zeros_like(self.W1)
        db1 = np.zeros_like(self.b1)
        dW2 = np.zeros_like(self.W2)
        db2 = np.zeros_like(self.b2)


        # Loop over each episode
        for log_prob, (x, h, logits), action in episode_log_probs:
            # Softmax again to recompute probabilities
            probs = softmax(logits)

            # dL/d_logits = -(e_a - probs) * G (policy gradient)
            d_logits = probs.copy()
            d_logits[action] -= 1.0
            d_logits *= -G # Gradient Ascent Direction

            # Backpropagagate through W2
            dW2 += np.outer(h, d_logits)
            db2 += d_logits

            # Backpropagagate into hidden
            dh = d_logits @ self.W2.T
            
            # ReLU derivative
            dh_pre = dh * (h > 0)

            # Backpropagagate into W2
            dW1 += np.outer(x, dh_pre)
            db1 += dh_pre

        # Gradient clipping to prevent exploding gradients
        for grad in [dW1, db1, dW2, db2]:
            np.clip(grad, -5, 5, out=grad)

        # Update the parameters by dividing by episode length to normalize
        self.W1 -= self.lr * dW1 / len(episode_log_probs)
        self.b1 -= self.lr * db1 / len(episode_log_probs)
        self.W2 -= self.lr * dW2 / len(episode_log_probs)
        self.b2 -= self.lr * db2 / len(episode_log_probs)


# Training

In [ ]:
def generate_cities(n=20, seed=42):
    # Uniform Random Coordinates in [0, 1]
    rng = np.random.RandomState(seed)
    return rng.rand(n, 2)

def run_episode(policy, cities):
    # Start at city 0
    n = len(cities)
    # Mark all cities not visited initially
    visited = np.zeros(n)
    current = 0
    visited[0] = 1
    tour = [0]
    log_probs_with_cache = []

    # Visit until all cities visited
    for _ in range(n - 1):
        action, log_prob, cache = policy.select_action(current, visited)
        log_probs_with_cache.append((log_prob, cache, action))
        visited[action] = 1
        tour.append(action)
        current = action

    # Compute Length
    length = tour_length(cities, tour)
    return tour, length, log_probs_with_cache


def greedy_tour(policy, cities):
    n = len(cities)
    visited = np.zeros(n)
    current = 0
    visited[0] = 1
    tour = [0]
    for _ in range(n - 1):
        probs, _ = policy.forward(current, visited)
        action = int(np.argmax(probs))
        visited[action] = 1
        tour.append(action)
        current = action
    return tour, tour_length(cities, tour)

# Training the Travelling Salesman
def train_tsp(n_episodes=5000, n_cities=20, lr=1e-3, baseline_decay=0.99):
    cities = generate_cities(n_cities)
    policy = PolicyNetwork(n_cities, hidden=64, lr=lr)

    tour_lengths = []
    baseline = None
    best_tour = None
    best_length = float('inf')

    for ep in range(n_episodes):
        tour, length, log_probs_cache = run_episode(policy, cities)

        # Moving Average Baseline reduces Variance
        if baseline is None:
            baseline = length
        else:
            baseline = baseline_decay * baseline + (1 - baseline_decay) * length

        # If tour is shorter than baseline, positive reward
        G = -(length - baseline)
        policy.update(log_probs_cache, G)

        # Track the tour and keep the best sampled result
        tour_lengths.append(length)
        if length < best_length:
            best_length = length
            best_tour = tour[:]

        # Evaluate for every 500 episodes
        if (ep + 1) % 500 == 0:
            g_tour, g_len = greedy_tour(policy, cities)
            print(f"  Ep {ep+1:5d} | sampled={length:.3f} | greedy={g_len:.3f} | best={best_length:.3f}")

    final_greedy_tour, final_greedy_len = greedy_tour(policy, cities)
    print(f"\nQ2 TSP Results:")
    print(f"  Best sampled tour length : {best_length:.4f}")
    print(f"  Final greedy tour length : {final_greedy_len:.4f}")

    # Plot tour
    plot_tour(cities, final_greedy_tour, final_greedy_len, "q2_tour.png", "Q2 TSP - Greedy Tour")

    # Plot training curve
    window = 200
    smoothed = np.convolve(tour_lengths, np.ones(window) / window, mode='valid')
    plt.figure(figsize=(8, 4))
    plt.plot(tour_lengths, alpha=0.2, color='blue', label='Sampled')
    plt.plot(range(window - 1, len(tour_lengths)), smoothed, color='red', label=f'{window}-ep avg')
    plt.xlabel("Episode"); plt.ylabel("Tour Length")
    plt.title("Q2 TSP - Training Progress")
    plt.legend(); plt.tight_layout()
    plt.savefig("q2_training.png", dpi=100)
    plt.close()

    return policy, cities, best_tour, best_length, tour_lengths

def plot_tour(cities, tour, length, fname, title):
    fig, ax = plt.subplots(figsize=(7, 7))
    n = len(tour)
    for i in range(n):
        a, b = cities[tour[i]], cities[tour[(i + 1) % n]]
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", color='steelblue', lw=1.5))
    for i, (x, y) in enumerate(cities):
        ax.plot(x, y, 'ro', markersize=8)
        ax.text(x + 0.01, y + 0.01, str(i), fontsize=8)
    ax.set_title(f"{title}\nTour Length = {length:.4f}")
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(fname, dpi=100)
    plt.close()

In [9]:
policy, cities, best_tour, best_len, history = train_tsp(n_episodes=5000)
print(f"Best tour: {best_tour}")
print(f"Saved: q2_tour.png, q2_training.png")

  Ep   500 | sampled=9.839 | greedy=11.891 | best=8.166
  Ep  1000 | sampled=10.869 | greedy=11.891 | best=8.166
  Ep  1500 | sampled=10.999 | greedy=12.640 | best=7.347
  Ep  2000 | sampled=8.545 | greedy=11.506 | best=7.347
  Ep  2500 | sampled=11.076 | greedy=11.506 | best=7.347
  Ep  3000 | sampled=12.790 | greedy=12.640 | best=7.347
  Ep  3500 | sampled=10.508 | greedy=11.506 | best=7.347
  Ep  4000 | sampled=9.481 | greedy=11.506 | best=7.347
  Ep  4500 | sampled=11.385 | greedy=11.506 | best=7.347
  Ep  5000 | sampled=11.780 | greedy=11.506 | best=7.347

Q2 TSP Results:
  Best sampled tour length : 7.3470
  Final greedy tour length : 11.5060
Best tour: [0, 19, 17, 12, 1, 4, 11, 7, 18, 2, 16, 13, 8, 10, 14, 6, 9, 15, 5, 3]
Saved: q2_tour.png, q2_training.png
